<a href="https://colab.research.google.com/github/thbieltraining/Calculadora-SENAI/blob/main/RECIS%C3%83O_CALC_SENAI_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from datetime import date, datetime, timedelta
from decimal import Decimal, ROUND_HALF_UP
from enum import Enum

# ================================================================
# CALCULADORA DE RESCISÃO SENAI 2026
# Módulo independente de FOPAG
# ================================================================

SALARIO_MINIMO_2026 = Decimal("1621.00")
DEDUCAO_DEPENDENTE_IRRF = Decimal("189.59")
DESCONTO_SIMPLIFICADO_IRRF = Decimal("607.20")
DOIS_CASAS = Decimal("0.01")

TABELA_INSS = [
    (Decimal("1621.00"), Decimal("0.075"), Decimal("0.00")),
    (Decimal("2902.84"), Decimal("0.09"), Decimal("24.31")),
    (Decimal("4304.87"), Decimal("0.12"), Decimal("111.40")),
    (Decimal("8475.55"), Decimal("0.14"), Decimal("197.50")),
]

TABELA_IRRF = [
    (Decimal("2428.80"), Decimal("0.00"), Decimal("0.00")),


    (Decimal("2826.65"), Decimal("0.075"), Decimal("182.16")),
    (Decimal("3751.05"), Decimal("0.15"), Decimal("394.16")),
    (Decimal("4664.68"), Decimal("0.225"), Decimal("675.49")),
    (Decimal("999999999999.99"), Decimal("0.275"), Decimal("908.73")),
]


class MotivoRescisao(Enum):
    SEM_JUSTA_CAUSA = "sem_justa_causa"
    PEDIDO_DEMISSAO = "pedido_demissao"
    JUSTA_CAUSA = "justa_causa"
    RESCISAO_INDIRETA = "rescisao_indireta"
    TERMINO_EXPERIENCIA = "termino_experiencia"
    ANTECIPACAO_EXPERIENCIA_EMPREGADOR = "antecipacao_experiencia_empregador"
    ANTECIPACAO_EXPERIENCIA_EMPREGADO = "antecipacao_experiencia_empregado"
    ACORDO_MUTUO = "acordo_mutuo"
    CULPA_RECIPROCA = "culpa_reciproca"


# ================================================================
# HELPERS
# ================================================================

def D(valor):
    return Decimal(str(valor))


def arredonda(valor):
    return D(valor).quantize(DOIS_CASAS, rounding=ROUND_HALF_UP)


def dinheiro(valor):
    return f"R$ {arredonda(valor):.2f}"


def ler_numero(mensagem, valor_padrao, tipo=Decimal):
    while True:
        entrada = input(f"{mensagem} [{valor_padrao}]: ").strip()
        if not entrada:
            return tipo(str(valor_padrao).replace(",", "."))
        try:
            return tipo(entrada.replace(",", "."))
        except ValueError:
            print("⚠️ Valor inválido. Digite novamente.")


def ler_sim_nao(mensagem, padrao="n"):
    while True:
        entrada = input(f"{mensagem} (s/n) [{padrao}]: ").strip().lower() or padrao
        if entrada in ("s", "sim"):
            return True
        if entrada in ("n", "nao", "não"):
            return False
        print("⚠️ Informe apenas S ou N.")


def ler_data(mensagem, padrao="01/01/2026"):
    while True:
        entrada = input(f"{mensagem} [{padrao}]: ").strip() or padrao
        entrada = entrada.replace("-", "/")
        try:
            return datetime.strptime(entrada, "%d/%m/%Y").date()
        except ValueError:
            print("⚠️ Data inválida. Use DD/MM/AAAA.")


# ================================================================
# TRIBUTAÇÃO
# ================================================================

def calc_inss(base):
    base = arredonda(max(D(base), Decimal("0.00")))
    for teto, aliquota, deducao in TABELA_INSS:
        if base <= teto:
            return max(arredonda(base * aliquota - deducao), Decimal("0.00"))
    return arredonda(
        TABELA_INSS[-1][0] * TABELA_INSS[-1][1] - TABELA_INSS[-1][2]
    )


def imposto_progressivo_irrf(base):
    base = arredonda(max(D(base), Decimal("0.00")))
    for teto, aliquota, deducao in TABELA_IRRF:
        if base <= teto:
            return max(arredonda(base * aliquota - deducao), Decimal("0.00"))
    return Decimal("0.00")


def calc_irrf_2026(rendimento_tributavel, deducoes_legais):
    """
    Compara deduções legais com o desconto simplificado de R$ 607,20.
    Aplica a redução mensal de 2026 sobre o imposto progressivo.
    """
    rendimento = arredonda(max(D(rendimento_tributavel), Decimal("0.00")))
    deducoes_legais = arredonda(max(D(deducoes_legais), Decimal("0.00")))

    base_legal = max(arredonda(rendimento - deducoes_legais), Decimal("0.00"))
    base_simplificada = max(
        arredonda(rendimento - DESCONTO_SIMPLIFICADO_IRRF),
        Decimal("0.00"),
    )

    imposto_legal = imposto_progressivo_irrf(base_legal)
    imposto_simplificado = imposto_progressivo_irrf(base_simplificada)

    if imposto_simplificado < imposto_legal:
        base_escolhida = base_simplificada
        imposto_antes_reducao = imposto_simplificado
        modalidade = "Desconto simplificado"
    else:
        base_escolhida = base_legal
        imposto_antes_reducao = imposto_legal
        modalidade = "Deduções legais"

    reducao = Decimal("0.00")
    if rendimento <= Decimal("5000.00"):
        reducao = min(imposto_antes_reducao, Decimal("312.89"))
    elif rendimento <= Decimal("7350.00"):
        reducao_calculada = arredonda(
            Decimal("978.62") - Decimal("0.133145") * rendimento
        )
        reducao = min(max(reducao_calculada, Decimal("0.00")), imposto_antes_reducao)

    imposto_final = max(arredonda(imposto_antes_reducao - reducao), Decimal("0.00"))

    return {
        "rendimento_tributavel": rendimento,
        "base_legal": base_legal,
        "base_simplificada": base_simplificada,
        "base_escolhida": base_escolhida,
        "modalidade": modalidade,
        "imposto_antes_reducao": imposto_antes_reducao,
        "reducao_2026": reducao,
        "imposto": imposto_final,
    }


# ================================================================
# REGRAS DE RESCISÃO
# ================================================================

def calcular_anos_completos(data_admissao, data_fim):
    anos = data_fim.year - data_admissao.year
    if (data_fim.month, data_fim.day) < (data_admissao.month, data_admissao.day):
        anos -= 1
    return max(0, anos)


def calcular_aviso_previo(data_admissao, data_demissao):
    anos = calcular_anos_completos(data_admissao, data_demissao)
    dias_adicionais = min(anos * 3, 60)
    return 30 + dias_adicionais, anos, dias_adicionais


def adicionar_anos_seguro(data_base, anos):
    try:
        return data_base.replace(year=data_base.year + anos)
    except ValueError:
        return data_base.replace(year=data_base.year + anos, day=28)


def calcular_avos_13(data_admissao, data_fim):
    inicio_ano = date(data_fim.year, 1, 1)
    inicio = max(data_admissao, inicio_ano)
    avos = 0
    cursor = inicio

    while cursor <= data_fim:
        ultimo_dia_mes = date(cursor.year, cursor.month, 1)
        if cursor.month == 12:
            proximo_mes = date(cursor.year + 1, 1, 1)
        else:
            proximo_mes = date(cursor.year, cursor.month + 1, 1)
        ultimo_dia_mes = proximo_mes - timedelta(days=1)

        inicio_contagem = max(cursor, date(cursor.year, cursor.month, 1))
        fim_contagem = min(data_fim, ultimo_dia_mes)
        dias = (fim_contagem - inicio_contagem).days + 1

        if dias >= 15:
            avos += 1

        cursor = proximo_mes

    return min(12, avos)


def calcular_avos_ferias(data_admissao, data_fim):
    inicio_periodo = adicionar_anos_seguro(
        data_admissao,
        max(0, calcular_anos_completos(data_admissao, data_fim)),
    )

    if inicio_periodo > data_fim:
        inicio_periodo = adicionar_anos_seguro(inicio_periodo, -1)

    avos = 0
    cursor = inicio_periodo

    while cursor < data_fim:
        proximo = adicionar_mes(cursor)
        fim = min(proximo, data_fim)
        dias = (fim - cursor).days
        if dias >= 15:
            avos += 1
        cursor = proximo

    return min(12, avos)


def adicionar_mes(data_base):
    if data_base.month == 12:
        ano, mes = data_base.year + 1, 1
    else:
        ano, mes = data_base.year, data_base.month + 1

    # Evita datas inválidas em meses menores.
    import calendar
    ultimo_dia = calendar.monthrange(ano, mes)[1]
    return date(ano, mes, min(data_base.day, ultimo_dia))


def calcular_media_variaveis(historico_mensal):
    """
    historico_mensal: lista de dicionários, por exemplo:
    [{"horas_extras": 350, "comissoes": 200, "adicionais": 100}, ...]

    A média é calculada sobre os meses informados.
    """
    if not historico_mensal:
        return {
            "meses": 0,
            "horas_extras": Decimal("0.00"),
            "comissoes": Decimal("0.00"),
            "adicionais": Decimal("0.00"),
            "total": Decimal("0.00"),
        }

    meses = len(historico_mensal)
    totais = {
        "horas_extras": Decimal("0.00"),
        "comissoes": Decimal("0.00"),
        "adicionais": Decimal("0.00"),
    }

    for mes in historico_mensal:
        for chave in totais:
            totais[chave] += D(mes.get(chave, 0))

    medias = {chave: arredonda(valor / meses) for chave, valor in totais.items()}
    medias["meses"] = meses
    medias["total"] = arredonda(sum(medias[chave] for chave in totais))
    return medias


def calcular_indenizacao_experiencia_empregador(remuneracao, data_demissao, data_termino):
    dias_restantes = max((data_termino - data_demissao).days, 0)
    valor_integral = arredonda(D(remuneracao) / Decimal("30") * dias_restantes)
    return {
        "dias_restantes": dias_restantes,
        "valor_base": valor_integral,
        "indenizacao_art_479": arredonda(valor_integral / Decimal("2")),
    }


def calcular_indenizacao_experiencia_empregado(remuneracao, data_demissao, data_termino, prejuizo_comprovado):
    dias_restantes = max((data_termino - data_demissao).days, 0)
    limite = arredonda(D(remuneracao) / Decimal("30") * dias_restantes)
    prejuizo = max(D(prejuizo_comprovado), Decimal("0.00"))
    return {
        "dias_restantes": dias_restantes,
        "limite_art_480": limite,
        "prejuizo_informado": arredonda(prejuizo),
        "desconto_art_480": min(arredonda(prejuizo), limite),
    }


def calcular_multa_art_477(salario, data_fim_contrato, data_pagamento, trabalhador_deu_causa=False):
    prazo = data_fim_contrato + timedelta(days=10)
    atraso = data_pagamento > prazo
    devida = atraso and not trabalhador_deu_causa
    return {
        "prazo_legal": prazo,
        "data_pagamento": data_pagamento,
        "houve_atraso": atraso,
        "trabalhador_deu_causa": trabalhador_deu_causa,
        "devida": devida,
        "valor": arredonda(D(salario)) if devida else Decimal("0.00"),
    }


def obter_regras_motivo(motivo):
    regras = {
        MotivoRescisao.SEM_JUSTA_CAUSA: {
            "aviso": True,
            "13": True,
            "ferias_proporcionais": True,
            "fgts_multa": Decimal("0.40"),
        },
        MotivoRescisao.PEDIDO_DEMISSAO: {
            "aviso": True,
            "13": True,
            "ferias_proporcionais": True,
            "fgts_multa": Decimal("0.00"),
        },
        MotivoRescisao.JUSTA_CAUSA: {
            "aviso": False,
            "13": False,
            "ferias_proporcionais": False,
            "fgts_multa": Decimal("0.00"),
        },
        MotivoRescisao.RESCISAO_INDIRETA: {
            "aviso": True,
            "13": True,
            "ferias_proporcionais": True,
            "fgts_multa": Decimal("0.40"),
        },
        MotivoRescisao.TERMINO_EXPERIENCIA: {
            "aviso": False,
            "13": True,
            "ferias_proporcionais": True,
            "fgts_multa": Decimal("0.00"),
        },
        MotivoRescisao.ANTECIPACAO_EXPERIENCIA_EMPREGADOR: {
            "aviso": False,
            "13": True,
            "ferias_proporcionais": True,
            "fgts_multa": Decimal("0.40"),
        },
        MotivoRescisao.ANTECIPACAO_EXPERIENCIA_EMPREGADO: {
            "aviso": False,
            "13": True,
            "ferias_proporcionais": True,
            "fgts_multa": Decimal("0.00"),
        },
        MotivoRescisao.ACORDO_MUTUO: {
            "aviso": True,
            "13": True,
            "ferias_proporcionais": True,
            "fgts_multa": Decimal("0.20"),
        },
        MotivoRescisao.CULPA_RECIPROCA: {
            "aviso": True,
            "13": True,
            "ferias_proporcionais": True,
            "fgts_multa": Decimal("0.20"),
            "fator_culpa_reciproca": Decimal("0.50"),
        },
    }
    return regras[motivo]


# ================================================================
# CORE — SOMENTE RESCISÃO
# ================================================================

def calcular_rescisao(dados):
    data_admissao = dados["data_admissao"]
    data_demissao = dados["data_demissao"]
    motivo = dados["motivo"]
    salario = D(dados["salario"])

    if data_demissao <= data_admissao:
        raise ValueError("A data de desligamento deve ser posterior à admissão.")
    if salario <= 0:
        raise ValueError("O salário deve ser maior que zero.")

    # CLT, art. 445, parágrafo único: contrato de experiência limitado a 90 dias.
    motivos_experiencia = (
        MotivoRescisao.TERMINO_EXPERIENCIA,
        MotivoRescisao.ANTECIPACAO_EXPERIENCIA_EMPREGADOR,
        MotivoRescisao.ANTECIPACAO_EXPERIENCIA_EMPREGADO,
    )
    if motivo in motivos_experiencia:
        data_limite_experiencia = data_admissao + timedelta(days=89)
        if motivo == MotivoRescisao.TERMINO_EXPERIENCIA and data_demissao > data_limite_experiencia:
            raise ValueError(
                "Contrato de experiência não pode ultrapassar 90 dias (CLT, art. 445). "
                "Verifique se o contrato passou a ser por prazo indeterminado."
            )
        data_fim_informada = dados.get("data_termino_experiencia")
        if data_fim_informada and data_fim_informada > data_limite_experiencia:
            raise ValueError(
                "A data prevista para término da experiência ultrapassa o limite legal de 90 dias (CLT, art. 445)."
            )

    regras = obter_regras_motivo(motivo)
    dependentes = int(dados.get("dependentes", 0))
    saldo_fgts = max(D(dados.get("saldo_fgts", 0)), Decimal("0.00"))

    # Remuneração + médias variáveis
    adicional = D(dados.get("valor_adicional", 0))
    media = calcular_media_variaveis(dados.get("historico_variaveis", []))
    remuneracao_base = arredonda(salario + adicional + media["total"])
    valor_dia = arredonda(remuneracao_base / Decimal("30"))

    # Aviso / projeção
    dias_aviso, anos_completos, dias_aviso_adic = calcular_aviso_previo(
        data_admissao, data_demissao
    )
    valor_aviso = Decimal("0.00")
    desconto_aviso = Decimal("0.00")
    tipo_aviso = "Não aplicável"
    data_fim_projetada = data_demissao

    if motivo in (MotivoRescisao.SEM_JUSTA_CAUSA, MotivoRescisao.RESCISAO_INDIRETA, MotivoRescisao.ACORDO_MUTUO, MotivoRescisao.CULPA_RECIPROCA):
        aviso_ind = dados.get("aviso_ind", True)
        if aviso_ind:
            fator = Decimal("0.50") if motivo in (MotivoRescisao.ACORDO_MUTUO, MotivoRescisao.CULPA_RECIPROCA) else Decimal("1.00")
            valor_aviso = arredonda(valor_dia * dias_aviso * fator)
            data_fim_projetada = data_demissao + timedelta(days=dias_aviso)
            tipo_aviso = f"Indenizado ({dias_aviso} dias)"
        else:
            tipo_aviso = f"Trabalhado ({dias_aviso} dias)"
    elif motivo == MotivoRescisao.PEDIDO_DEMISSAO:
        if dados.get("cumpriu_aviso", False):
            tipo_aviso = "Trabalhado"
        else:
            desconto_aviso = arredonda(valor_dia * min(dias_aviso, 30))
            tipo_aviso = "Descontado (30 dias)"

    # Saldo de salário
    dias_trabalhados = int(dados.get("dias_trabalhados_mes", data_demissao.day))
    faltas = int(dados.get("faltas_dias", 0))
    if dias_trabalhados < 0 or dias_trabalhados > 31:
        raise ValueError("Dias trabalhados inválidos.")
    if faltas < 0:
        raise ValueError("Faltas não podem ser negativas.")

    saldo_salario = arredonda(valor_dia * dias_trabalhados)
    desconto_faltas = arredonda(valor_dia * faltas)

    # Experiência
    indenizacao_art_479 = Decimal("0.00")
    desconto_art_480 = Decimal("0.00")
    experiencia_info = {}
    data_termino_experiencia = dados.get("data_termino_experiencia")

    if motivo == MotivoRescisao.ANTECIPACAO_EXPERIENCIA_EMPREGADOR:
        if not data_termino_experiencia:
            raise ValueError("Informe a data originalmente prevista para o término da experiência.")
        experiencia_info = calcular_indenizacao_experiencia_empregador(
            remuneracao_base, data_demissao, data_termino_experiencia
        )
        indenizacao_art_479 = experiencia_info["indenizacao_art_479"]

    elif motivo == MotivoRescisao.ANTECIPACAO_EXPERIENCIA_EMPREGADO:
        if not data_termino_experiencia:
            raise ValueError("Informe a data originalmente prevista para o término da experiência.")
        experiencia_info = calcular_indenizacao_experiencia_empregado(
            remuneracao_base,
            data_demissao,
            data_termino_experiencia,
            dados.get("prejuizo_comprovado", 0),
        )
        desconto_art_480 = experiencia_info["desconto_art_480"]

    # 13º e férias proporcionais consideram a projeção quando houver aviso indenizado.
    avos_13 = calcular_avos_13(data_admissao, data_fim_projetada) if regras["13"] else 0
    valor_13 = arredonda(remuneracao_base / Decimal("12") * avos_13)
    fator_culpa = regras.get("fator_culpa_reciproca", Decimal("1.00"))
    if motivo == MotivoRescisao.CULPA_RECIPROCA:
        valor_13 = arredonda(valor_13 * fator_culpa)

    avos_ferias = calcular_avos_ferias(data_admissao, data_fim_projetada) if regras["ferias_proporcionais"] else 0
    valor_ferias_prop = arredonda(remuneracao_base / Decimal("12") * avos_ferias)
    if motivo == MotivoRescisao.CULPA_RECIPROCA:
        valor_ferias_prop = arredonda(valor_ferias_prop * fator_culpa)
    terco_ferias_prop = arredonda(valor_ferias_prop / Decimal("3"))

    ferias_vencidas = int(dados.get("periodos_ferias_vencidas", 0))
    if ferias_vencidas < 0:
        raise ValueError("Quantidade de períodos vencidos inválida.")
    valor_ferias_vencidas = arredonda(remuneracao_base * ferias_vencidas)
    terco_ferias_vencidas = arredonda(valor_ferias_vencidas / Decimal("3"))

    # INSS/IRRF sobre parcelas tributáveis da rescisão.
    base_inss = max(
        arredonda(saldo_salario - desconto_faltas + valor_aviso + indenizacao_art_479),
        Decimal("0.00"),
    )
    inss = calc_inss(base_inss)
    irrf = calc_irrf_2026(
        base_inss,
        inss + Decimal(dependentes) * DEDUCAO_DEPENDENTE_IRRF,
    )

    # 13º é apurado separadamente para INSS/IRRF.
    inss_13 = calc_inss(valor_13)
    irrf_13 = calc_irrf_2026(
        valor_13,
        inss_13 + Decimal(dependentes) * DEDUCAO_DEPENDENTE_IRRF,
    )

    # FGTS e multa.
    fgts_mes = arredonda(base_inss * Decimal("0.08"))
    fgts_13 = arredonda(valor_13 * Decimal("0.08"))
    aliquota_multa = regras["fgts_multa"]
    multa_fgts = arredonda(saldo_fgts * aliquota_multa)

    # Art. 477: atraso é controlado explicitamente.
    multa_477 = calcular_multa_art_477(
        salario=remuneracao_base,
        data_fim_contrato=data_demissao,
        data_pagamento=dados.get("data_pagamento_rescisao", data_demissao + timedelta(days=1)),
        trabalhador_deu_causa=dados.get("trabalhador_deu_causa_atraso", False),
    )

    total_proventos = arredonda(
        saldo_salario
        + valor_aviso
        + indenizacao_art_479
        + valor_13
        + valor_ferias_vencidas
        + terco_ferias_vencidas
        + valor_ferias_prop
        + terco_ferias_prop
        + multa_477["valor"]
    )

    total_descontos = arredonda(
        desconto_faltas
        + desconto_aviso
        + desconto_art_480
        + inss
        + irrf["imposto"]
        + inss_13
        + irrf_13["imposto"]
    )

    liquido = arredonda(total_proventos - total_descontos)

    return {
        "motivo": motivo,
        "data_admissao": data_admissao,
        "data_demissao": data_demissao,
        "data_fim_projetada": data_fim_projetada,
        "anos_completos": anos_completos,
        "dias_aviso_totais": dias_aviso,
        "dias_aviso_adic": dias_aviso_adic,
        "tipo_aviso": tipo_aviso,
        "remuneracao_base": remuneracao_base,
        "media_variaveis": media,
        "saldo_salario": saldo_salario,
        "dias_trabalhados": dias_trabalhados,
        "desconto_faltas": desconto_faltas,
        "valor_aviso": valor_aviso,
        "desconto_aviso": desconto_aviso,
        "indenizacao_art_479": indenizacao_art_479,
        "desconto_art_480": desconto_art_480,
        "experiencia_info": experiencia_info,
        "avos_13": avos_13,
        "valor_13": valor_13,
        "inss_13": inss_13,
        "irrf_13": irrf_13,
        "avos_ferias": avos_ferias,
        "valor_ferias_prop": valor_ferias_prop,
        "terco_ferias_prop": terco_ferias_prop,
        "periodos_ferias_vencidas": ferias_vencidas,
        "valor_ferias_vencidas": valor_ferias_vencidas,
        "terco_ferias_vencidas": terco_ferias_vencidas,
        "base_inss": base_inss,
        "inss": inss,
        "irrf": irrf,
        "fgts_mes": fgts_mes,
        "fgts_13": fgts_13,
        "aliquota_multa_fgts": aliquota_multa,
        "multa_fgts": multa_fgts,
        "multa_art_477": multa_477,
        "total_proventos": total_proventos,
        "total_descontos": total_descontos,
        "liquido": liquido,
    }


# ================================================================
# INTERFACE TERMINAL — RESCISÃO
# ================================================================

def escolher_motivo():
    opcoes = {
        1: MotivoRescisao.SEM_JUSTA_CAUSA,
        2: MotivoRescisao.PEDIDO_DEMISSAO,
        3: MotivoRescisao.JUSTA_CAUSA,
        4: MotivoRescisao.RESCISAO_INDIRETA,
        5: MotivoRescisao.TERMINO_EXPERIENCIA,
        6: MotivoRescisao.ANTECIPACAO_EXPERIENCIA_EMPREGADOR,
        7: MotivoRescisao.ANTECIPACAO_EXPERIENCIA_EMPREGADO,
        8: MotivoRescisao.ACORDO_MUTUO,
        9: MotivoRescisao.CULPA_RECIPROCA,
    }
    print("\nMOTIVO DO DESLIGAMENTO")
    for numero, motivo in opcoes.items():
        print(f"{numero} = {motivo.value.replace('_', ' ').title()}")
    while True:
        opcao = ler_numero("Escolha o motivo", 1, int)
        if opcao in opcoes:
            return opcoes[opcao]
        print("⚠️ Opção inválida.")


def capturar_historico_variaveis():
    tem_variaveis = ler_sim_nao("Possui remuneração variável habitual para média?", "n")
    if not tem_variaveis:
        return []

    meses = ler_numero("Quantidade de meses de histórico para a média", 12, int)
    meses = max(1, min(meses, 12))
    historico = []

    print("\nInforme os valores por mês. Se não houver verba, use 0.")
    for i in range(1, meses + 1):
        print(f"Mês {i}/{meses}")
        historico.append({
            "horas_extras": ler_numero("  Horas extras / variáveis (R$)", "0.00"),
            "comissoes": ler_numero("  Comissões (R$)", "0.00"),
            "adicionais": ler_numero("  Outros adicionais variáveis (R$)", "0.00"),
        })
    return historico


def executar_calculadora_rescisao():
    print("=" * 72)
    print("             CALCULADORA DE RESCISÃO — SENAI 2026")
    print("             MÓDULO INDEPENDENTE DA FOPAG")
    print("=" * 72)

    data_admissao = ler_data("Data de Admissão", "01/02/2021")
    data_demissao = ler_data("Data de Desligamento", "15/09/2026")
    motivo = escolher_motivo()
    salario = ler_numero("Salário Fixo Mensal (R$)", "3200.00")

    tem_adicional = ler_sim_nao("Possui adicional de risco?", "n")
    valor_adicional = Decimal("0.00")
    if tem_adicional:
        tipo = input("Tipo [p=Periculosidade, i=Insalubridade] [p]: ").strip().lower() or "p"
        if tipo == "p":
            valor_adicional = arredonda(salario * Decimal("0.30"))
        elif tipo == "i":
            grau = ler_numero("Grau [1=10%, 2=20%, 3=40%]", 2, int)
            percentual = {1: Decimal("0.10"), 2: Decimal("0.20"), 3: Decimal("0.40")}.get(grau)
            if percentual is None:
                raise ValueError("Grau de insalubridade inválido.")
            valor_adicional = arredonda(SALARIO_MINIMO_2026 * percentual)
        else:
            raise ValueError("Tipo de adicional inválido.")

    dependentes = ler_numero("Dependentes para IRRF", 0, int)
    saldo_fgts = ler_numero("Saldo do FGTS para fins rescisórios (R$)", "12500.00")
    historico_variaveis = capturar_historico_variaveis()

    aviso_ind = False
    cumpriu_aviso = False
    if motivo in (MotivoRescisao.SEM_JUSTA_CAUSA, MotivoRescisao.RESCISAO_INDIRETA, MotivoRescisao.ACORDO_MUTUO, MotivoRescisao.CULPA_RECIPROCA):
        aviso_ind = ler_sim_nao("Aviso prévio será indenizado?", "s")
    elif motivo == MotivoRescisao.PEDIDO_DEMISSAO:
        cumpriu_aviso = ler_sim_nao("Empregado cumpriu o aviso prévio?", "n")

    faltas_dias = ler_numero("Faltas injustificadas no mês do desligamento", 0, int)
    dias_trabalhados = ler_numero("Dias efetivamente devidos no mês do desligamento", data_demissao.day, int)
    periodos_vencidos = ler_numero("Quantidade de períodos de férias vencidas", 0, int)

    data_termino_experiencia = None
    prejuizo_comprovado = Decimal("0.00")
    if motivo in (
        MotivoRescisao.TERMINO_EXPERIENCIA,
        MotivoRescisao.ANTECIPACAO_EXPERIENCIA_EMPREGADOR,
        MotivoRescisao.ANTECIPACAO_EXPERIENCIA_EMPREGADO,
    ):
        data_termino_experiencia = ler_data("Data originalmente prevista para término da experiência")
        if motivo == MotivoRescisao.TERMINO_EXPERIENCIA and data_demissao != data_termino_experiencia:
            print("⚠️ Atenção: no término normal, a data de desligamento deve corresponder ao término contratual informado.")
        if motivo == MotivoRescisao.ANTECIPACAO_EXPERIENCIA_EMPREGADO:
            prejuizo_comprovado = ler_numero("Prejuízo comprovado pelo empregador (R$)", "0.00")

    trabalhador_deu_causa = False
    data_pagamento = data_demissao + timedelta(days=1)
    quer_controlar_477 = ler_sim_nao("Deseja informar a data real de pagamento para verificar Art. 477?", "n")
    if quer_controlar_477:
        data_pagamento = ler_data("Data real do pagamento das verbas rescisórias")
        trabalhador_deu_causa = ler_sim_nao("O trabalhador deu causa comprovada ao atraso?", "n")

    dados = {
        "data_admissao": data_admissao,
        "data_demissao": data_demissao,
        "motivo": motivo,
        "salario": salario,
        "valor_adicional": valor_adicional,
        "dependentes": dependentes,
        "saldo_fgts": saldo_fgts,
        "historico_variaveis": historico_variaveis,
        "aviso_ind": aviso_ind,
        "cumpriu_aviso": cumpriu_aviso,
        "faltas_dias": faltas_dias,
        "dias_trabalhados_mes": dias_trabalhados,
        "periodos_ferias_vencidas": periodos_vencidos,
        "data_termino_experiencia": data_termino_experiencia,
        "prejuizo_comprovado": prejuizo_comprovado,
        "data_pagamento_rescisao": data_pagamento,
        "trabalhador_deu_causa_atraso": trabalhador_deu_causa,
    }

    res = calcular_rescisao(dados)

    print("\n" + "=" * 72)
    print("                 DEMONSTRATIVO DE RESCISÃO — TRCT")
    print("=" * 72)
    print(f"Motivo               : {res['motivo'].value.replace('_', ' ').title()}")
    if res["motivo"] == MotivoRescisao.CULPA_RECIPROCA:
        print("Regra aplicada       : CLT art. 484 e Súmula 14/TST (verbas específicas pela metade)")
    print(f"Admissão             : {res['data_admissao']:%d/%m/%Y}")
    print(f"Desligamento         : {res['data_demissao']:%d/%m/%Y}")
    print(f"Remuneração base     : {dinheiro(res['remuneracao_base'])}")
    print(f"Aviso                : {res['tipo_aviso']}")
    if res["data_fim_projetada"] != res["data_demissao"]:
        print(f"Projeção aviso       : {res['data_fim_projetada']:%d/%m/%Y}")

    print("-" * 72)
    print("PROVENTOS")
    print(f"Saldo de salário                    : {dinheiro(res['saldo_salario'])}")
    if res["valor_aviso"]:
        print(f"Aviso prévio indenizado             : {dinheiro(res['valor_aviso'])}")
    if res["indenizacao_art_479"]:
        print(f"Indenização Art. 479                : {dinheiro(res['indenizacao_art_479'])}")
    if res["valor_13"]:
        print(f"13º proporcional ({res['avos_13']}/12)       : {dinheiro(res['valor_13'])}")
    if res["valor_ferias_vencidas"]:
        print(f"Férias vencidas                     : {dinheiro(res['valor_ferias_vencidas'])}")
        print(f"1/3 férias vencidas                 : {dinheiro(res['terco_ferias_vencidas'])}")
    if res["valor_ferias_prop"]:
        print(f"Férias proporcionais ({res['avos_ferias']}/12) : {dinheiro(res['valor_ferias_prop'])}")
        print(f"1/3 férias proporcionais            : {dinheiro(res['terco_ferias_prop'])}")
    if res["multa_art_477"]["valor"]:
        print(f"Multa Art. 477                       : {dinheiro(res['multa_art_477']['valor'])}")
    print(f"TOTAL DE PROVENTOS                  : {dinheiro(res['total_proventos'])}")

    print("-" * 72)
    print("DESCONTOS")
    if res["desconto_faltas"]:
        print(f"Faltas injustificadas                : - {dinheiro(res['desconto_faltas'])}")
    if res["desconto_aviso"]:
        print(f"Aviso prévio                         : - {dinheiro(res['desconto_aviso'])}")
    if res["desconto_art_480"]:
        print(f"Indenização/compensação Art. 480     : - {dinheiro(res['desconto_art_480'])}")
    print(f"INSS                                 : - {dinheiro(res['inss'])}")
    print(f"IRRF                                 : - {dinheiro(res['irrf']['imposto'])}")
    print(f"INSS 13º                             : - {dinheiro(res['inss_13'])}")
    print(f"IRRF 13º                             : - {dinheiro(res['irrf_13']['imposto'])}")
    print(f"TOTAL DE DESCONTOS                   : - {dinheiro(res['total_descontos'])}")

    print("-" * 72)
    print(f"VALOR LÍQUIDO DA RESCISÃO             : {dinheiro(res['liquido'])}")

    print("\nMEMÓRIA FGTS")
    print(f"FGTS mês da rescisão                 : {dinheiro(res['fgts_mes'])}")
    print(f"FGTS sobre 13º                       : {dinheiro(res['fgts_13'])}")
    print(f"Multa FGTS ({res['aliquota_multa_fgts'] * 100:.0f}%)                 : {dinheiro(res['multa_fgts'])}")

    print("\nART. 477")
    print(f"Prazo legal de pagamento             : {res['multa_art_477']['prazo_legal']:%d/%m/%Y}")
    print(f"Pagamento informado                  : {res['multa_art_477']['data_pagamento']:%d/%m/%Y}")
    print(f"Atraso                                : {'SIM' if res['multa_art_477']['houve_atraso'] else 'NÃO'}")
    print(f"Multa devida                         : {'SIM' if res['multa_art_477']['devida'] else 'NÃO'}")
    print("=" * 72)


if __name__ == "__main__":
    try:
        executar_calculadora_rescisao()
    except ValueError as erro:
        print("\n" + "=" * 72)
        print("ERRO DE ENTRADA")
        print("=" * 72)
        print(f"⚠️ {erro}")
        print("=" * 72)


             CALCULADORA DE RESCISÃO — SENAI 2026
             MÓDULO INDEPENDENTE DA FOPAG
Data de Admissão [01/02/2021]: 
Data de Desligamento [15/09/2026]: 

MOTIVO DO DESLIGAMENTO
1 = Sem Justa Causa
2 = Pedido Demissao
3 = Justa Causa
4 = Rescisao Indireta
5 = Termino Experiencia
6 = Antecipacao Experiencia Empregador
7 = Antecipacao Experiencia Empregado
8 = Acordo Mutuo
9 = Culpa Reciproca
Escolha o motivo [1]: 
Salário Fixo Mensal (R$) [3200.00]: 
Possui adicional de risco? (s/n) [n]: 
Dependentes para IRRF [0]: 
Saldo do FGTS para fins rescisórios (R$) [12500.00]: 
Possui remuneração variável habitual para média? (s/n) [n]: 
Aviso prévio será indenizado? (s/n) [s]: 
Faltas injustificadas no mês do desligamento [0]: 
Dias efetivamente devidos no mês do desligamento [15]: 
Quantidade de períodos de férias vencidas [0]: 
Deseja informar a data real de pagamento para verificar Art. 477? (s/n) [n]: 

                 DEMONSTRATIVO DE RESCISÃO — TRCT
Motivo               : Sem Justa